In [1]:
import sys
print(sys.executable)

/Users/jairamdulasi/llm-zoomcamp-2026/.venv/bin/python


In [2]:
import sqlite3

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)

captured_spans = []


class CapturingConsoleExporter(ConsoleSpanExporter):
    def export(self, spans):
        captured_spans.extend(spans)
        return super().export(spans)


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True


provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(CapturingConsoleExporter()))
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from rag_helper import RAGBase


class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            return response

In [4]:
from dotenv import load_dotenv

load_dotenv()

from starter import index, client

assistant = RAGTraced(index=index, llm_client=client)

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = assistant.rag(query)
print(answer)

print(f"\nTotal spans: {len(captured_spans)}")
for s in captured_spans:
    duration_ms = (s.end_time - s.start_time) / 1e6
    print(f" - {s.name}  duration={duration_ms:.1f}ms  attrs={dict(s.attributes)}")

{
    "name": "search",
    "context": {
        "trace_id": "0x2cd83404e8c933045927dfda20cbbd32",
        "span_id": "0xc72d8c46b04b6616",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x388deccecf8417a7",
    "start_time": "2026-07-19T02:00:52.188549Z",
    "end_time": "2026-07-19T02:00:52.192674Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2324f833-b46c-49c8-b38f-47654180c3e6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x2cd83404e8c933045927dfda20cbbd32",
        "span_id": "0x195a95c65e2bb7b8",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [6]:
for _ in range(3):
    assistant.rag(query)

print("Done — 3 additional runs completed (4 total including Cell 4).")

{
    "name": "search",
    "context": {
        "trace_id": "0x9a5a33e8e9241969f45eb55bb2176206",
        "span_id": "0x915ad062c394d66a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1a727ef8e433b67c",
    "start_time": "2026-07-19T02:00:54.421368Z",
    "end_time": "2026-07-19T02:00:54.422763Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2324f833-b46c-49c8-b38f-47654180c3e6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x9a5a33e8e9241969f45eb55bb2176206",
        "span_id": "0xee52312665fa4580",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [7]:
import sqlite3

conn = sqlite3.connect("traces.db")
rows = conn.execute(
    "SELECT name, start_time, end_time, input_tokens, output_tokens FROM spans"
).fetchall()

print(f"Total rows in spans table: {len(rows)}")
for r in rows:
    print(r)

Total rows in spans table: 12
('search', 1784426452188549000, 1784426452192674000, None, None)
('llm', 1784426452193962000, 1784426454414614000, 7111, 110)
('rag', 1784426452188512000, 1784426454416277000, None, None)
('search', 1784426454421368000, 1784426454422763000, None, None)
('llm', 1784426454423555000, 1784426455894565000, 7111, 95)
('rag', 1784426454421303000, 1784426455896798000, None, None)
('search', 1784426455897816000, 1784426455898776000, None, None)
('llm', 1784426455899766000, 1784426457447536000, 7111, 95)
('rag', 1784426455897792000, 1784426457451468000, None, None)
('search', 1784426457452769000, 1784426457454568000, None, None)
('llm', 1784426457456282000, 1784426458880744000, 7111, 124)
('rag', 1784426457452740000, 1784426458885565000, None, None)


In [8]:
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1e6

print(df[df["name"] != "rag"].groupby("name")["duration_ms"].sum())

name
llm       6663.894
search       8.279
Name: duration_ms, dtype: float64


In [ ]:
llm_rows = df[df["name"] == "llm"][["start_time", "input_tokens"]].sort_values("start_time")
print(llm_rows)

stats = llm_rows["input_tokens"].agg(["min", "max", "mean"])
print(stats)
pct_spread = (stats["max"] - stats["min"]) / stats["mean"] * 100
print(f"\nSpread: {pct_spread:.1f}% of mean")

             start_time  input_tokens
1   1784426452193962000        7111.0
4   1784426454423555000        7111.0
7   1784426455899766000        7111.0
10  1784426457456282000        7111.0
min     7111.0
max     7111.0
mean    7111.0
Name: input_tokens, dtype: float64

Spread: 0.0% of mean
